In [1]:
import os
os.environ["MPLBACKEND"] = "TkAgg"  # Or "agg" if no GUI

import matplotlib
import matplotlib.pyplot as plt

print("Matplotlib version:", matplotlib.__version__)
print("Backend:", plt.get_backend())


Matplotlib version: 3.10.3
Backend: TkAgg


In [2]:
# Import and verify library versions
import numpy as np
import pandas as pd
from scipy.fft import fft, fftfreq
import os
import h5py
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, resample_poly, find_peaks
import neurokit2 as nk
import importlib
import matplotlib.pyplot as plt 

print("NeuroKit2 version:", nk.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)


# Pandas display settings
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)  # Better to use None

# Print library versions
print("NeuroKit2 version:", nk.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)





NeuroKit2 version: 0.2.10
NumPy version: 1.26.4
Pandas version: 2.2.3
NeuroKit2 version: 0.2.10
NumPy version: 1.26.4
Pandas version: 2.2.3


In [3]:
# RESP FILE PATHS
RI1_3_6_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s3_6_p5_3_nRB3_20250621_125312_merged.h5"
RI2_3_6_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s3_6_p5_3_nRB3_20250621_131158_merged.h5"
BLRI_s3_6_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s3_6_p5_3_nRB3_20250621_123634_merged.h5"
BLRI_s4_7_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\BLRI_s4_7_p5_2_nRB3_20250621_144806_merged.h5"
RI_4_7_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI1_s4_7_p5_2_nRB3_20250621_150707_merged.h5"
RI2_4_7_h5_path = r"E:\Aim1\AIM1\Day1_new\resp_h5\RI2_s4_7_p5_2_nRB3_20250621_152519_merged.h5"

# BORIS CSVs (no comma at the end or it makes a tuple)
RI1_3_6_boris = pd.read_csv(r"E:\Aim1\AIM1\Day1_new\boris\RI1_s3_6_p5_3_nRB3_HEEPS.csv")
RI2_3_6_boris = pd.read_csv(r"E:\Aim1\AIM1\Day1_new\boris\RI2_s3_6_p_5_3_nRB3_2025062.csv")
RI1_4_7_boris = pd.read_csv(r"E:\Aim1\AIM1\Day1_new\boris\RI1_s4_7_p5_2_nRB3_HEEPS.csv")
RI2_4_7_boris = pd.read_csv(r"E:\Aim1\AIM1\Day1_new\boris\RI2_s4_7_p5_2_nRB3_HEEPS.csv")


In [20]:
results = [get_kit(f) for f in file_paths]


NameError: name 'file_paths' is not defined

In [4]:
import h5py

def explore_h5_structure(h5_path):
    with h5py.File(h5_path, 'r') as f:
        print("Available top-level keys:", list(f.keys()))
        for key in f.keys():
            print(f"\nContents of '{key}':")
            print(f[key])


In [5]:
explore_h5_structure(RI1_3_6_h5_path)


Available top-level keys: ['ekg', 'ekg_metadata', 'metadata', 'resp', 'resp_metadata']

Contents of 'ekg':
<HDF5 dataset "ekg": shape (12065446,), type "<i2">

Contents of 'ekg_metadata':
<HDF5 group "/ekg_metadata" (0 members)>

Contents of 'metadata':
<HDF5 group "/metadata" (0 members)>

Contents of 'resp':
<HDF5 dataset "resp": shape (12065446, 1), type "<f8">

Contents of 'resp_metadata':
<HDF5 group "/resp_metadata" (0 members)>


In [12]:
def load_resp_signal(h5_path):
    with h5py.File(h5_path, 'r') as f:
        print("Available keys:", list(f.keys()))  # check for correct dataset name
        # Often it's under something like 'Respiration' or 'AI_Resp'
        signal = np.array(f[list(f.keys())[0]])
    return signal


In [13]:
raw_resp = load_resp_signal(RI1_3_6_h5_path)
print("Length:", len(raw_resp))


Available keys: ['ekg', 'ekg_metadata', 'metadata', 'resp', 'resp_metadata']
Length: 12065446


In [8]:
# Downsample to 100 Hz
resp_downsampled = nk.signal_resample(raw_resp, sampling_rate=1017, desired_sampling_rate=100)


In [9]:
signals, info = nk.rsp_process(resp_downsampled, sampling_rate=100)
nk.rsp_plot(signals, info)


In [11]:
print(resp_downsampled.shape)
print(np.isnan(resp_downsampled).sum())  # Should be 0


(1186376,)
0


In [18]:
import h5py
import numpy as np
import neurokit2 as nk
import matplotlib.pyplot as plt

# === Function to load RESP signal ===
def load_resp_signal(h5_path):
    with h5py.File(h5_path, 'r') as f:
        print("Available keys:", list(f.keys()))  # Show what's inside
        signal = np.array(f['resp'])  # <- Use the actual respiration key
    return signal

# === Load raw RESP signal ===
raw_resp = load_resp_signal(RI1_3_6_h5_path)
print("Raw signal length:", len(raw_resp))

# Flatten to 1D
raw_resp = raw_resp.reshape(-1)
print("Fixed shape:", raw_resp.shape)

# Downsample
resp_downsampled = nk.signal_resample(raw_resp, sampling_rate=1017, desired_sampling_rate=100)
print("Downsampled shape:", resp_downsampled.shape)

# Process and plot
signals, info = nk.rsp_process(resp_downsampled, sampling_rate=100)
nk.rsp_plot(signals, info)
plt.show()



Available keys: ['ekg', 'ekg_metadata', 'metadata', 'resp', 'resp_metadata']
Raw signal length: 12065446
Fixed shape: (12065446,)
Downsampled shape: (1186376,)


In [ ]:
signals


In [19]:
RSP_Rate = signals['RSP_Rate']

In [ ]:
RSP_Rate

In [16]:
print("Raw shape:", raw_resp.shape)


Raw shape: (12065446, 1)


In [17]:
print("Squeezed shape:", raw_resp.shape)


Squeezed shape: (12065446, 1)
